In [10]:
import json
import re
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# =============================================================================
# 0) PATHS
# =============================================================================

INPUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
MAIN_DATASET_FILENAME = "MainDataset.csv"
OUTPUT_DIR = INPUT_DIR / "0.2-validation_outputs"

MAIN_DATASET_PATH = INPUT_DIR / MAIN_DATASET_FILENAME


# =============================================================================
# 1) SETTINGS
# =============================================================================

ALLOWED_STYLES = {"Community", "Custom", "GMD", "Third-Party"}
COMPLETE_VERDICTS = {"success", "failure"}
KNOWN_OTHER_VERDICTS = {
    "cancelled",
    "skipped",
    "timed_out",
    "neutral",
    "action_required",
    "stale",
    "startup_failure",
    "",
}

EPSILON_SEC = 1.0
EXPECTED_MAINDATASET_ROWS = 8906
EXPECTED_BASE_ROWS = 7771
ENFORCE_SNAPSHOT_COUNTS = True

# Whether to try cross-file step validation if step telemetry is found
ENABLE_STEP_CROSSCHECK = True


# =============================================================================
# 2) FIELD DEFINITIONS
# =============================================================================

COLUMN_ALIASES: Dict[str, List[str]] = {
    # identifiers
    "run_id": ["run_id", "id_run", "gha_run_id"],
    "style": ["style", "execution_style", "emu_style", "style_name"],
    "repo": ["repo_full_name", "repo", "repository", "repository_full_name"],
    "workflow_id": ["workflow_id", "gha_workflow_id"],
    "workflow_name": ["workflow_name", "name_workflow"],

    # timestamps
    "run_start": ["run_started_at", "started_at_run", "run_start", "created_at"],
    "run_end": ["run_completed_at", "completed_at_run", "run_end", "updated_at"],
    "trigger_time": ["run_triggered_at", "triggered_at", "queued_at"],

    # global durations
    "run_duration_sec": [
        "run_duration_seconds",
        "run_duration_sec",
        "duration_run_seconds",
        "run_duration",
    ],
    "queue_time_sec": ["queue_time_seconds", "queue_time_sec"],

    # controller source
    "instru_job_count": ["instru_job_count", "instrumentation_job_count", "instr_job_count"],

    # layer 1 boundaries
    "l1_first_instr_job_start": [
        "first_instr_job_started_at",
        "first_instru_job_started_at",
        "first_instrumentation_job_started_at",
    ],
    "l1_last_instr_job_end": [
        "last_instr_job_completed_at",
        "last_instru_job_completed_at",
        "last_instrumentation_job_completed_at",
    ],

    # layer 1 durations
    "l1_pre_sec": [
        "time_to_instrumentation_envelope_seconds",
        "time_to_instrumentation_envelope_sec",
        "ttie_seconds",
        "ttie_sec",
    ],
    "l1_mid_sec": [
        "instrumentation_job_envelope_seconds",
        "instrumentation_job_envelope_sec",
        "ije_seconds",
        "ije_sec",
    ],
    "l1_post_sec": [
        "post_instrumentation_tail_seconds",
        "post_instrumentation_tail_sec",
        "pit_seconds",
        "pit_sec",
    ],

    # layer 2 boundaries
    "l2_invocation_start": [
        "matched_invocation_step_started_at",
        "invocation_step_started_at",
        "l2_invocation_start",
    ],
    "l2_exec_end": [
        "last_execution_related_step_completed_at",
        "execution_related_step_end_at",
        "l2_exec_end",
    ],

    # layer 2 durations
    "l2_pre_sec": ["pre_invocation_seconds", "pre_invocation_sec"],
    "l2_mid_sec": ["invocation_execution_window_seconds", "invocation_execution_window_sec"],
    "l2_post_sec": ["post_invocation_seconds", "post_invocation_sec"],

    # controllers
    "instrumentation_executed": [
        "instrumentation_executed",
        "instru_executed",
        "executed_instrumentation",
    ],
    "attempt_num": ["run_attempt", "attempt_num", "attempt"],
    "run_verdict": ["run_conclusion", "run_verdict", "conclusion_run"],
    "instr_verdict": ["instru_conclusion", "instrumentation_conclusion", "instrumentation_verdict"],

    # grouping duration aggregates
    "g_setup_sec": ["grp_setup_duration_seconds", "setup_duration_seconds"],
    "g_provision_sec": ["grp_provision_duration_seconds", "provision_duration_seconds"],
    "g_test_sec": ["grp_test_duration_seconds", "test_duration_seconds"],
    "g_artifact_sec": ["grp_artifact_report_duration_seconds", "artifact_report_duration_seconds"],
    "g_cleanup_sec": ["grp_cleanup_teardown_duration_seconds", "cleanup_teardown_duration_seconds"],
    "g_other_sec": ["grp_other_duration_seconds", "other_duration_seconds"],

    "g_exec_related_sec": [
        "grp_execution_related_duration_seconds",
        "execution_related_duration_seconds",
    ],
    "g_non_exec_sec": [
        "grp_non_execution_overhead_duration_seconds",
        "non_execution_overhead_duration_seconds",
    ],

    "g_pretest_sec": [
        "grp_pre_test_overhead_duration_seconds",
        "pre_test_overhead_duration_seconds",
    ],
    "g_activetest_sec": [
        "grp_active_test_duration_seconds",
        "active_test_duration_seconds",
    ],
    "g_posttest_sec": [
        "grp_post_test_overhead_duration_seconds",
        "post_test_overhead_duration_seconds",
    ],

    # optional step counts
    "c_setup": ["grp_setup_step_count", "setup_step_count"],
    "c_provision": ["grp_provision_step_count", "provision_step_count"],
    "c_test": ["grp_test_step_count", "test_step_count"],
    "c_artifact": ["grp_artifact_report_step_count", "artifact_report_step_count"],
    "c_cleanup": ["grp_cleanup_teardown_step_count", "cleanup_teardown_step_count"],
    "c_other": ["grp_other_step_count", "other_step_count"],
}

COLUMN_PATTERNS: Dict[str, List[List[str]]] = {
    "run_id": [["run", "id"]],
    "style": [["style"]],
    "repo": [["repo"], ["repository"]],
    "workflow_id": [["workflow", "id"]],
    "workflow_name": [["workflow", "name"]],

    "run_start": [["run", "start"], ["run", "started"]],
    "run_end": [["run", "end"], ["run", "complete"], ["run", "completed"]],
    "trigger_time": [["trigger"], ["queued"]],

    "run_duration_sec": [["run", "duration"]],
    "queue_time_sec": [["queue", "time"]],

    "instru_job_count": [["instru", "job", "count"], ["instrumentation", "job", "count"]],

    "l1_first_instr_job_start": [["first", "instr", "job", "start"], ["first", "instrumentation", "job", "start"]],
    "l1_last_instr_job_end": [["last", "instr", "job", "end"], ["last", "instrumentation", "job", "end"]],

    "l1_pre_sec": [["time", "instrumentation", "envelope"]],
    "l1_mid_sec": [["instrumentation", "job", "envelope"]],
    "l1_post_sec": [["post", "instrumentation", "tail"]],

    "l2_invocation_start": [["invocation", "step", "start"], ["matched", "invocation", "start"]],
    "l2_exec_end": [["execution", "related", "end"], ["last", "execution", "step", "end"]],

    "l2_pre_sec": [["pre", "invocation"]],
    "l2_mid_sec": [["invocation", "execution", "window"]],
    "l2_post_sec": [["post", "invocation"]],

    "instrumentation_executed": [["instrumentation", "executed"], ["instru", "executed"]],
    "attempt_num": [["attempt"]],
    "run_verdict": [["run", "conclusion"], ["run", "verdict"]],
    "instr_verdict": [["instru", "conclusion"], ["instrumentation", "conclusion"], ["instrumentation", "verdict"]],

    "g_setup_sec": [["setup", "duration"]],
    "g_provision_sec": [["provision", "duration"]],
    "g_test_sec": [["test", "duration"]],
    "g_artifact_sec": [["artifact", "report", "duration"], ["artifact", "duration"]],
    "g_cleanup_sec": [["cleanup", "teardown", "duration"], ["cleanup", "duration"]],
    "g_other_sec": [["other", "duration"]],

    "g_exec_related_sec": [["execution", "related", "duration"]],
    "g_non_exec_sec": [["non", "execution", "overhead", "duration"]],

    "g_pretest_sec": [["pre", "test", "overhead", "duration"]],
    "g_activetest_sec": [["active", "test", "duration"]],
    "g_posttest_sec": [["post", "test", "overhead", "duration"]],

    "c_setup": [["setup", "count"]],
    "c_provision": [["provision", "count"]],
    "c_test": [["test", "count"]],
    "c_artifact": [["artifact", "report", "count"], ["artifact", "count"]],
    "c_cleanup": [["cleanup", "teardown", "count"], ["cleanup", "count"]],
    "c_other": [["other", "count"]],
}


# =============================================================================
# 3) HELPERS
# =============================================================================

@dataclass
class RuleResult:
    rule_name: str
    category: str
    applicable_n: int
    passed_n: int
    failed_n: int
    skipped: bool
    skip_reason: str = ""
    notes: str = ""


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


def token_set(name: str) -> set:
    return set(normalize_name(name).split("_"))


def resolve_columns(df: pd.DataFrame) -> Dict[str, Optional[str]]:
    cols = list(df.columns)
    cols_lower = {c.lower(): c for c in cols}
    cols_norm = {normalize_name(c): c for c in cols}
    resolved: Dict[str, Optional[str]] = {}

    for key, candidates in COLUMN_ALIASES.items():
        resolved[key] = None

        # Exact alias matching
        for cand in candidates:
            if cand in cols:
                resolved[key] = cand
                break
            if cand.lower() in cols_lower:
                resolved[key] = cols_lower[cand.lower()]
                break
            cand_norm = normalize_name(cand)
            if cand_norm in cols_norm:
                resolved[key] = cols_norm[cand_norm]
                break

        if resolved[key] is not None:
            continue

        # Pattern matching fallback
        patterns = COLUMN_PATTERNS.get(key, [])
        for col in cols:
            col_tokens = token_set(col)
            matched = False
            for patt in patterns:
                if all(tok in col_tokens for tok in patt):
                    resolved[key] = col
                    matched = True
                    break
            if matched:
                break

    return resolved


def has_col(df: pd.DataFrame, colmap: Dict[str, Optional[str]], key: str) -> bool:
    col = colmap.get(key)
    return bool(col) and col in df.columns


def get_col(df: pd.DataFrame, colmap: Dict[str, Optional[str]], key: str) -> pd.Series:
    col = colmap.get(key)
    if col is None or col not in df.columns:
        raise KeyError(f"Column not resolved for key={key!r}")
    return df[col]


def parse_dt(s: pd.Series) -> pd.Series:
    """
    Parse consistently and strip timezone awareness so comparisons never mix
    tz-aware and tz-naive values.
    """
    dt = pd.to_datetime(s, errors="coerce", utc=True)
    return dt.dt.tz_localize(None)


def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def to_boolish(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    lowered = s.astype(str).str.strip().str.lower()
    return lowered.isin({"1", "true", "yes", "y"})


def abs_diff(a: pd.Series, b: pd.Series) -> pd.Series:
    return (a - b).abs()


def unresolved_keys(colmap: Dict[str, Optional[str]], keys: List[str]) -> List[str]:
    return [k for k in keys if not colmap.get(k)]


def combine_notna(df: pd.DataFrame, num: Dict[str, pd.Series], keys: List[str]) -> pd.Series:
    applicable = pd.Series(True, index=df.index)
    for k in keys:
        applicable &= num[k].notna()
    return applicable


def add_rule(
    rules: List[RuleResult],
    failures: List[Dict[str, Any]],
    df: pd.DataFrame,
    colmap: Dict[str, Optional[str]],
    rule_name: str,
    category: str,
    applicable: Optional[pd.Series],
    passed: Optional[pd.Series],
    notes: str = "",
    detail: str = "",
    extra_keys: Optional[List[str]] = None,
    skip_reason: str = "",
) -> None:
    if applicable is None or passed is None:
        rules.append(
            RuleResult(
                rule_name=rule_name,
                category=category,
                applicable_n=0,
                passed_n=0,
                failed_n=0,
                skipped=True,
                skip_reason=skip_reason,
                notes=notes,
            )
        )
        return

    applicable_n = int(applicable.sum())
    passed_n = int((applicable & passed).sum())
    failed_n = int((applicable & ~passed).sum())

    rules.append(
        RuleResult(
            rule_name=rule_name,
            category=category,
            applicable_n=applicable_n,
            passed_n=passed_n,
            failed_n=failed_n,
            skipped=False,
            notes=notes,
        )
    )

    fail_mask = applicable & ~passed
    if failed_n > 0:
        cols_to_keep = []
        for k in ["run_id", "style", "repo", "workflow_id"]:
            c = colmap.get(k)
            if c and c in df.columns and c not in cols_to_keep:
                cols_to_keep.append(c)

        if extra_keys:
            for k in extra_keys:
                c = colmap.get(k)
                if c and c in df.columns and c not in cols_to_keep:
                    cols_to_keep.append(c)

        sample = df.loc[fail_mask, cols_to_keep].copy()
        sample["rule_name"] = rule_name
        sample["category"] = category
        sample["detail"] = detail
        failures.extend(sample.to_dict(orient="records"))


def compute_derived_fields(
    df: pd.DataFrame,
    colmap: Dict[str, Optional[str]],
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    df2 = df.copy()
    derived_map: Dict[str, str] = {}

    # Derive run_duration_sec if possible
    if not has_col(df2, colmap, "run_duration_sec") and has_col(df2, colmap, "run_start") and has_col(df2, colmap, "run_end"):
        rs = parse_dt(get_col(df2, colmap, "run_start"))
        re = parse_dt(get_col(df2, colmap, "run_end"))
        df2["_derived_run_duration_sec"] = (re - rs).dt.total_seconds()
        colmap["run_duration_sec"] = "_derived_run_duration_sec"
        derived_map["run_duration_sec"] = "_derived_run_duration_sec"

    # Derive instrumentation_executed if possible
    if not has_col(df2, colmap, "instrumentation_executed") and has_col(df2, colmap, "instru_job_count"):
        s = to_num(get_col(df2, colmap, "instru_job_count"))
        df2["_derived_instrumentation_executed"] = s > 0
        colmap["instrumentation_executed"] = "_derived_instrumentation_executed"
        derived_map["instrumentation_executed"] = "_derived_instrumentation_executed"

    return df2, derived_map


def find_candidate_files(input_dir: Path) -> Dict[str, Optional[Path]]:
    files = list(input_dir.glob("*"))
    file_map = {
        "main_dataset": None,
        "step_csv": None,
        "step_zip": None,
        "verified_workflows": None,
        "url_list": None,
    }

    for f in files:
        name = f.name.lower()
        if file_map["main_dataset"] is None and name == MAIN_DATASET_FILENAME.lower():
            file_map["main_dataset"] = f
        if file_map["step_csv"] is None and "step" in name and f.suffix.lower() == ".csv":
            file_map["step_csv"] = f
        if file_map["step_zip"] is None and "step" in name and f.suffix.lower() == ".zip":
            file_map["step_zip"] = f
        if file_map["verified_workflows"] is None and "verified_workflows" in name and f.suffix.lower() == ".csv":
            file_map["verified_workflows"] = f
        if file_map["url_list"] is None and "url_list" in name and f.suffix.lower() == ".csv":
            file_map["url_list"] = f

    return file_map


def load_step_telemetry(file_map: Dict[str, Optional[Path]]) -> Tuple[Optional[pd.DataFrame], str]:
    # Prefer direct CSV
    if file_map.get("step_csv") and file_map["step_csv"].exists():
        try:
            return pd.read_csv(file_map["step_csv"], low_memory=False), f"loaded_csv:{file_map['step_csv'].name}"
        except Exception as e:
            return None, f"step_csv_load_failed:{e}"

    # Then ZIP containing CSV
    if file_map.get("step_zip") and file_map["step_zip"].exists():
        try:
            with zipfile.ZipFile(file_map["step_zip"], "r") as zf:
                csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
                if not csv_names:
                    return None, "step_zip_has_no_csv"
                # use first csv
                with zf.open(csv_names[0]) as f:
                    return pd.read_csv(f, low_memory=False), f"loaded_zip_csv:{file_map['step_zip'].name}::{csv_names[0]}"
        except Exception as e:
            return None, f"step_zip_load_failed:{e}"

    return None, "no_step_telemetry_found"


def validate_against_steps(
    main_df: pd.DataFrame,
    main_colmap: Dict[str, Optional[str]],
    step_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Optional cross-file checks. Conservative and schema-tolerant.
    """
    step_colmap = resolve_columns(step_df)

    results = []
    failures = []

    if not (has_col(main_df, main_colmap, "run_id") and has_col(step_df, step_colmap, "run_id")):
        return (
            pd.DataFrame([{
                "check_name": "step_crosscheck_run_presence",
                "status": "skipped",
                "notes": "run_id unresolved in main or step telemetry"
            }]),
            pd.DataFrame()
        )

    main_runs = set(get_col(main_df, main_colmap, "run_id").dropna().astype(str).unique())
    step_runs = set(get_col(step_df, step_colmap, "run_id").dropna().astype(str).unique())

    overlap = len(main_runs & step_runs)
    only_main = len(main_runs - step_runs)
    only_step = len(step_runs - main_runs)

    results.append({
        "check_name": "step_crosscheck_run_presence",
        "status": "executed",
        "main_unique_runs": len(main_runs),
        "step_unique_runs": len(step_runs),
        "overlap_runs": overlap,
        "main_only_runs": only_main,
        "step_only_runs": only_step,
        "notes": "Cross-file run overlap between MainDataset and step telemetry"
    })

    # Optional step-derived coverage check for invocation/execution-like fields
    for logical_field in ["l2_invocation_start", "l2_exec_end"]:
        if has_col(main_df, main_colmap, logical_field):
            present_n = int(get_col(main_df, main_colmap, logical_field).notna().sum())
            results.append({
                "check_name": f"{logical_field}_main_presence",
                "status": "executed",
                "present_rows": present_n,
                "notes": f"MainDataset presence count for {logical_field}"
            })

    return pd.DataFrame(results), pd.DataFrame(failures)


# =============================================================================
# 4) VALIDATION
# =============================================================================

def validate_main_dataset(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any], Dict[str, Optional[str]], Dict[str, str], pd.DataFrame]:
    colmap = resolve_columns(df)
    df, derived_map = compute_derived_fields(df, colmap)

    rules: List[RuleResult] = []
    failures: List[Dict[str, Any]] = []

    # -------------------------------------------------------------------------
    # structural consistency
    # -------------------------------------------------------------------------
    needed = ["run_id", "style"]
    if all(has_col(df, colmap, k) for k in needed):
        applicable = pd.Series(True, index=df.index)
        passed = ~df.duplicated(subset=[colmap["run_id"], colmap["style"]], keep=False)
        add_rule(
            rules, failures, df, colmap,
            "unique_run_style_key", "structural_consistency",
            applicable, passed,
            notes="Each run×style record should be unique.",
            detail="Duplicate run×style key detected.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "unique_run_style_key", "structural_consistency", None, None,
                 skip_reason=f"Unresolved columns: {unresolved_keys(colmap, needed)}")

    needed = ["run_id", "style", "run_start"]
    if all(has_col(df, colmap, k) for k in needed):
        req_cols = [colmap[k] for k in needed]
        applicable = pd.Series(True, index=df.index)
        passed = ~df[req_cols].isna().any(axis=1)
        add_rule(
            rules, failures, df, colmap,
            "required_minimum_core_fields_present", "structural_consistency",
            applicable, passed,
            notes=f"Required minimum columns: {req_cols}",
            detail="Missing one or more minimum core fields.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "required_minimum_core_fields_present", "structural_consistency", None, None,
                 skip_reason=f"Unresolved columns: {unresolved_keys(colmap, needed)}")

    needed = ["run_id", "style", "run_start", "run_end", "run_duration_sec"]
    if all(has_col(df, colmap, k) for k in needed):
        req_cols = [colmap[k] for k in needed]
        applicable = pd.Series(True, index=df.index)
        passed = ~df[req_cols].isna().any(axis=1)
        add_rule(
            rules, failures, df, colmap,
            "required_full_timing_core_fields_present", "structural_consistency",
            applicable, passed,
            notes=f"Full timing-core columns: {req_cols}",
            detail="Missing one or more full timing-core fields.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "required_full_timing_core_fields_present", "structural_consistency", None, None,
                 skip_reason=f"Unresolved or non-materialized columns: {unresolved_keys(colmap, needed)}")

    if has_col(df, colmap, "style"):
        s = get_col(df, colmap, "style")
        applicable = s.notna()
        passed = s.isin(ALLOWED_STYLES)
        add_rule(
            rules, failures, df, colmap,
            "style_in_allowed_scope", "structural_consistency",
            applicable, passed,
            notes=f"Allowed styles: {sorted(ALLOWED_STYLES)}",
            detail="Style outside four-style scope.",
            extra_keys=["style"],
        )

    # -------------------------------------------------------------------------
    # temporal ordering
    # -------------------------------------------------------------------------
    dt = {}
    for k in ["run_start", "run_end", "trigger_time", "l1_first_instr_job_start", "l1_last_instr_job_end", "l2_invocation_start", "l2_exec_end"]:
        if has_col(df, colmap, k):
            dt[k] = parse_dt(get_col(df, colmap, k))

    needed = ["run_start", "run_end"]
    if {"run_start", "run_end"}.issubset(dt):
        applicable = dt["run_start"].notna() & dt["run_end"].notna()
        passed = dt["run_start"] <= dt["run_end"]
        add_rule(
            rules, failures, df, colmap,
            "run_start_before_or_equal_run_end", "temporal_ordering",
            applicable, passed,
            detail="Run start occurs after run end.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "run_start_before_or_equal_run_end", "temporal_ordering", None, None,
                 skip_reason=f"Unresolved or non-materialized columns: {unresolved_keys(colmap, needed)}")

    if {"trigger_time", "run_start"}.issubset(dt):
        applicable = dt["trigger_time"].notna() & dt["run_start"].notna()
        passed = dt["trigger_time"] <= dt["run_start"]
        add_rule(
            rules, failures, df, colmap,
            "trigger_before_or_equal_run_start", "temporal_ordering",
            applicable, passed,
            notes="Optional non-study-facing sanity check.",
            detail="Trigger time occurs after run start.",
            extra_keys=["trigger_time", "run_start"],
        )
    else:
        add_rule(rules, failures, df, colmap, "trigger_before_or_equal_run_start", "temporal_ordering", None, None,
                 skip_reason="Trigger time not materialized/resolved.")

    needed = ["run_start", "l1_first_instr_job_start", "l1_last_instr_job_end", "run_end"]
    if {"run_start", "l1_first_instr_job_start", "l1_last_instr_job_end", "run_end"}.issubset(dt):
        applicable = (
            dt["run_start"].notna()
            & dt["l1_first_instr_job_start"].notna()
            & dt["l1_last_instr_job_end"].notna()
            & dt["run_end"].notna()
        )
        passed = (
            (dt["run_start"] <= dt["l1_first_instr_job_start"])
            & (dt["l1_first_instr_job_start"] <= dt["l1_last_instr_job_end"])
            & (dt["l1_last_instr_job_end"] <= dt["run_end"])
        )
        add_rule(
            rules, failures, df, colmap,
            "layer1_boundary_ordering_valid", "temporal_ordering",
            applicable, passed,
            detail="Layer 1 boundaries out of order.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "layer1_boundary_ordering_valid", "temporal_ordering", None, None,
                 skip_reason=f"Unresolved or non-materialized columns: {unresolved_keys(colmap, needed)}")

    needed = ["run_start", "l2_invocation_start", "l2_exec_end", "run_end"]
    if {"run_start", "l2_invocation_start", "l2_exec_end", "run_end"}.issubset(dt):
        applicable = (
            dt["run_start"].notna()
            & dt["l2_invocation_start"].notna()
            & dt["l2_exec_end"].notna()
            & dt["run_end"].notna()
        )
        passed = (
            (dt["run_start"] <= dt["l2_invocation_start"])
            & (dt["l2_invocation_start"] <= dt["l2_exec_end"])
            & (dt["l2_exec_end"] <= dt["run_end"])
        )
        add_rule(
            rules, failures, df, colmap,
            "layer2_boundary_ordering_valid", "temporal_ordering",
            applicable, passed,
            notes="Applied where Layer 2 boundaries are materialized.",
            detail="Layer 2 boundaries out of order.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "layer2_boundary_ordering_valid", "temporal_ordering", None, None,
                 skip_reason=f"Unresolved or non-materialized columns: {unresolved_keys(colmap, needed)}")

    # -------------------------------------------------------------------------
    # numeric / duration checks
    # -------------------------------------------------------------------------
    numeric_keys = [
        "run_duration_sec", "queue_time_sec", "instru_job_count", "attempt_num",
        "l1_pre_sec", "l1_mid_sec", "l1_post_sec",
        "l2_pre_sec", "l2_mid_sec", "l2_post_sec",
        "g_setup_sec", "g_provision_sec", "g_test_sec",
        "g_artifact_sec", "g_cleanup_sec", "g_other_sec",
        "g_exec_related_sec", "g_non_exec_sec",
        "g_pretest_sec", "g_activetest_sec", "g_posttest_sec",
        "c_setup", "c_provision", "c_test", "c_artifact", "c_cleanup", "c_other",
    ]
    num = {}
    for k in numeric_keys:
        if has_col(df, colmap, k):
            num[k] = to_num(get_col(df, colmap, k))

    for k in numeric_keys:
        if k in num:
            applicable = num[k].notna()
            passed = num[k] >= 0
            add_rule(
                rules, failures, df, colmap,
                f"{k}_non_negative", "duration_consistency",
                applicable, passed,
                detail=f"Negative value in {k}.",
                extra_keys=[k],
            )

    needed = ["run_duration_sec", "l1_pre_sec", "l1_mid_sec", "l1_post_sec"]
    if all(k in num for k in needed):
        applicable = combine_notna(df, num, needed)
        passed = abs_diff(
            num["l1_pre_sec"] + num["l1_mid_sec"] + num["l1_post_sec"],
            num["run_duration_sec"]
        ) <= EPSILON_SEC
        add_rule(
            rules, failures, df, colmap,
            "layer1_components_sum_to_run_duration", "duration_consistency",
            applicable, passed,
            notes=f"Tolerance = {EPSILON_SEC} sec.",
            detail="Layer 1 parts do not sum to run duration.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "layer1_components_sum_to_run_duration", "duration_consistency", None, None,
                 skip_reason=f"Unresolved or non-materialized fields: {needed}")

    needed = ["run_duration_sec", "l2_pre_sec", "l2_mid_sec", "l2_post_sec"]
    if all(k in num for k in needed):
        applicable = combine_notna(df, num, needed)
        passed = abs_diff(
            num["l2_pre_sec"] + num["l2_mid_sec"] + num["l2_post_sec"],
            num["run_duration_sec"]
        ) <= EPSILON_SEC
        add_rule(
            rules, failures, df, colmap,
            "layer2_components_sum_to_run_duration", "duration_consistency",
            applicable, passed,
            notes=f"Tolerance = {EPSILON_SEC} sec.",
            detail="Layer 2 parts do not sum to run duration.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "layer2_components_sum_to_run_duration", "duration_consistency", None, None,
                 skip_reason=f"Unresolved or non-materialized fields: {needed}")

    for k in ["l1_pre_sec", "l1_mid_sec", "l1_post_sec", "l2_pre_sec", "l2_mid_sec", "l2_post_sec"]:
        if "run_duration_sec" in num and k in num:
            applicable = num["run_duration_sec"].notna() & num[k].notna()
            passed = num[k] <= num["run_duration_sec"] + EPSILON_SEC
            add_rule(
                rules, failures, df, colmap,
                f"{k}_not_greater_than_run_duration", "duration_consistency",
                applicable, passed,
                detail=f"{k} exceeds run duration.",
                extra_keys=["run_duration_sec", k],
            )

    activity_keys = ["g_setup_sec", "g_provision_sec", "g_test_sec", "g_artifact_sec", "g_cleanup_sec", "g_other_sec"]
    needed = activity_keys + ["l2_pre_sec", "l2_mid_sec", "l2_post_sec"]
    if all(k in num for k in needed):
        applicable = combine_notna(df, num, needed)
        passed = abs_diff(
            sum(num[k] for k in activity_keys),
            num["l2_pre_sec"] + num["l2_mid_sec"] + num["l2_post_sec"]
        ) <= EPSILON_SEC
        add_rule(
            rules, failures, df, colmap,
            "activity_group_sum_matches_layer2_total", "cross_level_consistency",
            applicable, passed,
            notes=f"Tolerance = {EPSILON_SEC} sec.",
            detail="Activity-group sum does not match Layer 2 total.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "activity_group_sum_matches_layer2_total", "cross_level_consistency", None, None,
                 skip_reason=f"Unresolved or non-materialized fields: {needed}")

    needed = ["g_exec_related_sec", "g_non_exec_sec", "l2_pre_sec", "l2_mid_sec", "l2_post_sec"]
    if all(k in num for k in needed):
        applicable = combine_notna(df, num, needed)
        passed = abs_diff(
            num["g_exec_related_sec"] + num["g_non_exec_sec"],
            num["l2_pre_sec"] + num["l2_mid_sec"] + num["l2_post_sec"]
        ) <= EPSILON_SEC
        add_rule(
            rules, failures, df, colmap,
            "execution_role_sum_matches_layer2_total", "cross_level_consistency",
            applicable, passed,
            notes=f"Tolerance = {EPSILON_SEC} sec.",
            detail="Execution-role sum does not match Layer 2 total.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "execution_role_sum_matches_layer2_total", "cross_level_consistency", None, None,
                 skip_reason=f"Unresolved or non-materialized fields: {needed}")

    needed = ["g_pretest_sec", "g_activetest_sec", "g_posttest_sec", "l2_pre_sec", "l2_mid_sec", "l2_post_sec"]
    if all(k in num for k in needed):
        applicable = combine_notna(df, num, needed)
        passed = abs_diff(
            num["g_pretest_sec"] + num["g_activetest_sec"] + num["g_posttest_sec"],
            num["l2_pre_sec"] + num["l2_mid_sec"] + num["l2_post_sec"]
        ) <= EPSILON_SEC
        add_rule(
            rules, failures, df, colmap,
            "overhead_phase_sum_matches_layer2_total", "cross_level_consistency",
            applicable, passed,
            notes=f"Tolerance = {EPSILON_SEC} sec.",
            detail="Overhead-phase sum does not match Layer 2 total.",
            extra_keys=needed,
        )
    else:
        add_rule(rules, failures, df, colmap, "overhead_phase_sum_matches_layer2_total", "cross_level_consistency", None, None,
                 skip_reason=f"Unresolved or non-materialized fields: {needed}")

    # -------------------------------------------------------------------------
    # controller consistency
    # -------------------------------------------------------------------------
    if "attempt_num" in num:
        applicable = num["attempt_num"].notna()
        passed = num["attempt_num"] >= 1
        add_rule(
            rules, failures, df, colmap,
            "attempt_number_valid", "controller_consistency",
            applicable, passed,
            detail="Attempt number < 1.",
            extra_keys=["attempt_num"],
        )
    else:
        add_rule(rules, failures, df, colmap, "attempt_number_valid", "controller_consistency", None, None,
                 skip_reason="Attempt number not materialized/resolved.")

    if has_col(df, colmap, "run_verdict"):
        v = get_col(df, colmap, "run_verdict").astype(str).str.strip().str.lower()
        applicable = get_col(df, colmap, "run_verdict").notna()
        passed = v.isin(COMPLETE_VERDICTS | KNOWN_OTHER_VERDICTS)
        add_rule(
            rules, failures, df, colmap,
            "run_verdict_value_recognized", "controller_consistency",
            applicable, passed,
            detail="Unexpected run verdict.",
            extra_keys=["run_verdict"],
        )
    else:
        add_rule(rules, failures, df, colmap, "run_verdict_value_recognized", "controller_consistency", None, None,
                 skip_reason="Run verdict not materialized/resolved.")

    if has_col(df, colmap, "instr_verdict"):
        v = get_col(df, colmap, "instr_verdict").astype(str).str.strip().str.lower()
        applicable = get_col(df, colmap, "instr_verdict").notna()
        passed = v.isin(COMPLETE_VERDICTS | KNOWN_OTHER_VERDICTS)
        add_rule(
            rules, failures, df, colmap,
            "instrumentation_verdict_value_recognized", "controller_consistency",
            applicable, passed,
            detail="Unexpected instrumentation verdict.",
            extra_keys=["instr_verdict"],
        )
    else:
        add_rule(rules, failures, df, colmap, "instrumentation_verdict_value_recognized", "controller_consistency", None, None,
                 skip_reason="Instrumentation verdict not materialized/resolved.")

    if has_col(df, colmap, "instrumentation_executed"):
        b = to_boolish(get_col(df, colmap, "instrumentation_executed"))
        applicable = get_col(df, colmap, "instrumentation_executed").notna()
        passed = b
        add_rule(
            rules, failures, df, colmap,
            "main_dataset_is_instrumentation_executed_only", "controller_consistency",
            applicable, passed,
            notes="Expected for MainDataset; derived from instru_job_count if needed.",
            detail="MainDataset contains instrumentation_executed = false rows.",
            extra_keys=["instrumentation_executed"],
        )
    else:
        add_rule(rules, failures, df, colmap, "main_dataset_is_instrumentation_executed_only", "controller_consistency", None, None,
                 skip_reason="Instrumentation-executed field unresolved and not derivable.")

    # -------------------------------------------------------------------------
    # coverage and schema coverage
    # -------------------------------------------------------------------------
    coverage_rows: List[Dict[str, Any]] = []
    n_total = int(len(df))
    coverage_rows.append({"metric": "records_total", "value": n_total, "notes": "Total MainDataset rows."})

    if has_col(df, colmap, "style"):
        coverage_rows.append({
            "metric": "records_in_four_style_scope",
            "value": int(get_col(df, colmap, "style").isin(ALLOWED_STYLES).sum()),
            "notes": "Rows in four-style scope."
        })

    if all(has_col(df, colmap, k) for k in ["l1_pre_sec", "l1_mid_sec", "l1_post_sec"]):
        coverage_rows.append({
            "metric": "records_with_complete_layer1_durations",
            "value": int((get_col(df, colmap, "l1_pre_sec").notna() & get_col(df, colmap, "l1_mid_sec").notna() & get_col(df, colmap, "l1_post_sec").notna()).sum()),
            "notes": "Rows with complete Layer 1 durations."
        })
    else:
        coverage_rows.append({
            "metric": "records_with_complete_layer1_durations",
            "value": None,
            "notes": "Layer 1 fields not fully materialized/resolved."
        })

    if all(has_col(df, colmap, k) for k in ["l2_pre_sec", "l2_mid_sec", "l2_post_sec"]):
        coverage_rows.append({
            "metric": "records_with_complete_layer2_durations",
            "value": int((get_col(df, colmap, "l2_pre_sec").notna() & get_col(df, colmap, "l2_mid_sec").notna() & get_col(df, colmap, "l2_post_sec").notna()).sum()),
            "notes": "Rows with complete Layer 2 durations."
        })
    else:
        coverage_rows.append({
            "metric": "records_with_complete_layer2_durations",
            "value": None,
            "notes": "Layer 2 fields not fully materialized/resolved."
        })

    if has_col(df, colmap, "run_id") and has_col(df, colmap, "style"):
        c = df.groupby(colmap["run_id"])[colmap["style"]].nunique(dropna=True)
        coverage_rows.append({"metric": "single_style_runs", "value": int((c == 1).sum()), "notes": "Runs with one style."})
        coverage_rows.append({"metric": "multi_style_runs", "value": int((c > 1).sum()), "notes": "Runs with more than one style."})

    base_mask = pd.Series(True, index=df.index)
    if has_col(df, colmap, "style"):
        base_mask &= get_col(df, colmap, "style").isin(ALLOWED_STYLES)
    if has_col(df, colmap, "instrumentation_executed"):
        base_mask &= to_boolish(get_col(df, colmap, "instrumentation_executed"))
    if has_col(df, colmap, "attempt_num"):
        base_mask &= to_num(get_col(df, colmap, "attempt_num")).eq(1)
    if has_col(df, colmap, "run_verdict"):
        base_mask &= get_col(df, colmap, "run_verdict").astype(str).str.strip().str.lower().isin(COMPLETE_VERDICTS)
    if has_col(df, colmap, "instr_verdict"):
        base_mask &= get_col(df, colmap, "instr_verdict").astype(str).str.strip().str.lower().isin(COMPLETE_VERDICTS)

    base_n = int(base_mask.sum())
    coverage_rows.append({"metric": "base_subset_records", "value": base_n, "notes": "Controller-defined Base subset."})

    if ENFORCE_SNAPSHOT_COUNTS:
        coverage_rows.extend([
            {"metric": "expected_maindataset_rows", "value": EXPECTED_MAINDATASET_ROWS, "notes": "Methodology snapshot expectation."},
            {"metric": "maindataset_row_delta_vs_expected", "value": n_total - EXPECTED_MAINDATASET_ROWS, "notes": "Observed minus expected."},
            {"metric": "expected_base_rows", "value": EXPECTED_BASE_ROWS, "notes": "Methodology snapshot expectation."},
            {"metric": "base_row_delta_vs_expected", "value": base_n - EXPECTED_BASE_ROWS, "notes": "Observed minus expected."},
        ])

    schema_rows = []
    for key in COLUMN_ALIASES.keys():
        resolved = colmap.get(key)
        status = "resolved" if resolved else "unresolved"
        origin = "derived" if resolved and str(resolved).startswith("_derived_") else ("materialized" if resolved else "")
        schema_rows.append({
            "logical_field": key,
            "resolved_column": resolved,
            "status": status,
            "origin": origin,
        })
    schema_df = pd.DataFrame(schema_rows)

    coverage_df = pd.DataFrame(coverage_rows)
    rules_df = pd.DataFrame([r.__dict__ for r in rules])
    if not rules_df.empty:
        rules_df["pass_rate"] = np.where(
            rules_df["applicable_n"] > 0,
            rules_df["passed_n"] / rules_df["applicable_n"],
            np.nan,
        )
    failures_df = pd.DataFrame(failures)

    summary = {
        "dataset_rows": n_total,
        "rules_total": int(len(rules)),
        "rules_executed": int((~rules_df["skipped"]).sum()) if not rules_df.empty else 0,
        "rules_skipped": int((rules_df["skipped"]).sum()) if not rules_df.empty else 0,
        "failures_total": int(len(failures)),
        "base_subset_records": base_n,
        "expected_maindataset_rows": EXPECTED_MAINDATASET_ROWS,
        "expected_base_rows": EXPECTED_BASE_ROWS,
        "epsilon_sec": EPSILON_SEC,
        "resolved_columns": colmap,
        "derived_fields": derived_map,
        "resolved_field_count": int((schema_df["status"] == "resolved").sum()),
        "unresolved_field_count": int((schema_df["status"] == "unresolved").sum()),
    }

    return rules_df, failures_df, coverage_df, summary, colmap, derived_map, schema_df


# =============================================================================
# 5) RUN + SAVE OUTPUTS
# =============================================================================

def run_validation(input_dir: Path, main_dataset_path: Path, output_dir: Path) -> Dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    file_map = find_candidate_files(input_dir)

    df = pd.read_csv(main_dataset_path, low_memory=False)
    rules_df, failures_df, coverage_df, summary, colmap, derived_map, schema_df = validate_main_dataset(df)

    # Optional step telemetry cross-check
    step_check_df = pd.DataFrame()
    step_fail_df = pd.DataFrame()
    step_source_note = "step_crosscheck_not_attempted"
    if ENABLE_STEP_CROSSCHECK:
        step_df, step_source_note = load_step_telemetry(file_map)
        if step_df is not None:
            step_check_df, step_fail_df = validate_against_steps(df, colmap, step_df)

    # Paths
    rules_path = output_dir / "validation_rule_summary.csv"
    failures_path = output_dir / "validation_failures.csv"
    coverage_path = output_dir / "validation_coverage_summary.csv"
    summary_path = output_dir / "validation_record_summary.json"
    resolved_columns_path = output_dir / "validation_resolved_columns.json"
    derived_fields_path = output_dir / "validation_derived_fields.json"
    schema_coverage_path = output_dir / "validation_schema_coverage.csv"
    input_columns_path = output_dir / "validation_input_columns.csv"
    file_inventory_path = output_dir / "validation_file_inventory.json"
    step_check_path = output_dir / "validation_step_crosscheck.csv"
    step_fail_path = output_dir / "validation_step_crosscheck_failures.csv"

    # Save outputs
    rules_df.to_csv(rules_path, index=False)
    failures_df.to_csv(failures_path, index=False)
    coverage_df.to_csv(coverage_path, index=False)
    schema_df.to_csv(schema_coverage_path, index=False)
    pd.DataFrame({"input_column_name": list(df.columns)}).to_csv(input_columns_path, index=False)
    step_check_df.to_csv(step_check_path, index=False)
    step_fail_df.to_csv(step_fail_path, index=False)

    summary["step_crosscheck_source"] = step_source_note

    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    with resolved_columns_path.open("w", encoding="utf-8") as f:
        json.dump(colmap, f, indent=2)

    with derived_fields_path.open("w", encoding="utf-8") as f:
        json.dump(derived_map, f, indent=2)

    with file_inventory_path.open("w", encoding="utf-8") as f:
        json.dump({k: (str(v) if v else None) for k, v in file_map.items()}, f, indent=2)

    print("Validation completed.")
    print(f"Input rows:         {len(df)}")
    print(f"Rules total:        {summary['rules_total']}")
    print(f"Rules ran:          {summary['rules_executed']}")
    print(f"Rules skipped:      {summary['rules_skipped']}")
    print(f"Failures:           {summary['failures_total']}")
    print(f"Base subset:        {summary['base_subset_records']}")
    print(f"Resolved fields:    {summary['resolved_field_count']}")
    print(f"Unresolved fields:  {summary['unresolved_field_count']}")
    print(f"Step cross-check:   {step_source_note}")
    print()
    print(f"Rule summary:        {rules_path}")
    print(f"Failures log:        {failures_path}")
    print(f"Coverage summary:    {coverage_path}")
    print(f"Summary JSON:        {summary_path}")
    print(f"Resolved columns:    {resolved_columns_path}")
    print(f"Derived fields:      {derived_fields_path}")
    print(f"Schema coverage:     {schema_coverage_path}")
    print(f"Input columns audit: {input_columns_path}")
    print(f"File inventory:      {file_inventory_path}")
    print(f"Step cross-check:    {step_check_path}")
    print(f"Step cross failures: {step_fail_path}")

    return {
        "rule_summary_path": str(rules_path),
        "failures_path": str(failures_path),
        "coverage_path": str(coverage_path),
        "summary_json_path": str(summary_path),
        "resolved_columns_path": str(resolved_columns_path),
        "derived_fields_path": str(derived_fields_path),
        "schema_coverage_path": str(schema_coverage_path),
        "input_columns_audit_path": str(input_columns_path),
        "file_inventory_path": str(file_inventory_path),
        "step_crosscheck_path": str(step_check_path),
        "step_crosscheck_failures_path": str(step_fail_path),
        "summary": summary,
    }


# =============================================================================
# 6) EXECUTE
# =============================================================================

result = run_validation(
    input_dir=INPUT_DIR,
    main_dataset_path=MAIN_DATASET_PATH,
    output_dir=OUTPUT_DIR,
)

result


Validation completed.
Input rows:         8906
Rules total:        38
Rules ran:          32
Rules skipped:      6
Failures:           211
Base subset:        7771
Resolved fields:    23
Unresolved fields:  19
Step cross-check:   loaded_csv:run_steps_v16_stage3_breakdown.csv

Rule summary:        C:\Android Mobile App\ICST2026_Ext\0.2-validation_outputs\validation_rule_summary.csv
Failures log:        C:\Android Mobile App\ICST2026_Ext\0.2-validation_outputs\validation_failures.csv
Coverage summary:    C:\Android Mobile App\ICST2026_Ext\0.2-validation_outputs\validation_coverage_summary.csv
Summary JSON:        C:\Android Mobile App\ICST2026_Ext\0.2-validation_outputs\validation_record_summary.json
Resolved columns:    C:\Android Mobile App\ICST2026_Ext\0.2-validation_outputs\validation_resolved_columns.json
Derived fields:      C:\Android Mobile App\ICST2026_Ext\0.2-validation_outputs\validation_derived_fields.json
Schema coverage:     C:\Android Mobile App\ICST2026_Ext\0.2-validation

{'rule_summary_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_rule_summary.csv',
 'failures_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_failures.csv',
 'coverage_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_coverage_summary.csv',
 'summary_json_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_record_summary.json',
 'resolved_columns_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_resolved_columns.json',
 'derived_fields_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_derived_fields.json',
 'schema_coverage_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_schema_coverage.csv',
 'input_columns_audit_path': 'C:\\Android Mobile App\\ICST2026_Ext\\0.2-validation_outputs\\validation_input_columns.csv',
 'file_inventory_path': 'C:\\Android Mobile App\\ICST2026